# Lectures 19-23: Portfolio Workflow in Colab

This notebook combines the portfolio lessons into one Colab-ready workflow:

- Lecture 19: Build the first portfolio in Colab
- Lecture 20: Update the portfolio and review version history
- Lecture 21: Localize the portfolio
- Lecture 22: Final review of the human view and agent-ready YAML
- Lecture 23: Wrap-up, portfolio workflow recap, and next steps

Run the cells from top to bottom. The notebook creates a single workspace folder, builds a portfolio from source lanes, refreshes it after a source change, creates localized HTML pages, validates the machine-readable YAML, and closes with the course wrap-up.

### What We Just Covered

The earlier commands taught the individual parts: validation, generation, fragments, catalogs, and graphs. Lectures 16 and 17 connect those parts to the portfolio builder: a repeatable workflow that takes source material and produces human-reviewable HTML plus machine-readable YAML.

Next, we run that full portfolio workflow in Colab.

# Lecture 19: Build the First Portfolio in Colab

This is the main hands-on portfolio lesson. It loads source material, runs the portfolio builder, inspects generated folders, opens HTML output, and reviews catalog and graph files.

### What We Just Covered

The portfolio builder turns source lanes into a connected portfolio workspace. The source lanes represent objectives, use cases, signals, and product material; the outputs include generated YAML, ODPC catalog content, ODPG graph content, and an HTML review page.

Next, we create the Colab workspace, add source files, and run the first portfolio build.

## Prepare Colab

Install the SDK in the Colab runtime. **Note! We use now more fresh version.** The SDK is developing faster than I can create courses. Re-run this cell if Colab restarts the session.

In [1]:
!python -m pip install --upgrade open-data-products==0.3.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.7/373.7 kB 7.3 MB/s eta 0:00:00


## Store The Provider Key

If you use Claude, store `ANTHROPIC_API_KEY` in Colab secrets, then read it into the notebook environment.

Do not commit or share notebooks that contain real API keys.

### Before You Run This

The portfolio builder uses an LLM provider for generation and localization. Keep the API key in Colab secrets, not inside the notebook text. If the key is missing, provider-backed cells will print a skip message instead of failing without context.

What the code does:

- Tries to read `ANTHROPIC_API_KEY` from Colab secrets.
- Falls back to an existing environment variable when the notebook is not running in Colab.
- Stores the key as an environment variable so the SDK can use it in later cells.
- Prints whether provider-backed portfolio cells can run in this session.


In [2]:
import os

try:
    from google.colab import userdata
    key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    key = os.environ.get("ANTHROPIC_API_KEY")

if key:
    os.environ["ANTHROPIC_API_KEY"] = key
    print("ANTHROPIC_API_KEY is available for this runtime.")
else:
    print("ANTHROPIC_API_KEY is not set. Add it to Colab secrets before running provider-backed portfolio cells.")

ANTHROPIC_API_KEY is available for this runtime.


## Create A Workspace Folder

Keep the portfolio section in one working folder. In Colab, this keeps source files, generated YAML, HTML, localized pages, and version history together instead of scattering files through downloads.

The rest of Lectures 18-22 assume your current folder is this workspace.

### Before You Run This

The portfolio workflow creates many related files: source documents, generated YAML, HTML pages, localized pages, and version snapshots. Keeping them in one Colab workspace folder makes the later review and zip steps much easier.

What the code does:

- Creates `/content/odp-portfolio-workspace` if it does not already exist.
- Changes the active notebook folder to that workspace so all later files are created in one place.


In [3]:
!mkdir -p /content/odp-portfolio-workspace
%cd /content/odp-portfolio-workspace

/content/odp-portfolio-workspace


## Create Source Lanes

Create source folders in the notebook runtime. These four lanes match the input map from Lecture 17: objectives, use cases, signals, and products.

### Before You Run This

Source lanes are simple folders that tell the builder what kind of material it is reading. Objectives, use cases, signals, and product notes each play a different role in the final portfolio.

What the code does:

- Creates four source-lane folders.
- Gives the portfolio builder separate places for objectives, use cases, signals, and product notes.


In [4]:
!mkdir -p source_docs/objectives source_docs/use-cases source_docs/signals source_docs/products
!ls -al source_docs

total 24
drwxr-xr-x 6 root root 4096 Jun 25 12:58 .
drwxr-xr-x 3 root root 4096 Jun 25 12:58 ..
drwxr-xr-x 2 root root 4096 Jun 25 12:58 objectives
drwxr-xr-x 2 root root 4096 Jun 25 12:58 products
drwxr-xr-x 2 root root 4096 Jun 25 12:58 signals
drwxr-xr-x 2 root root 4096 Jun 25 12:58 use-cases


## Add Source Files

Run this shell cell inside `/content/odp-portfolio-workspace` to add one source file to each lane.

In [5]:
%%bash
cat > source_docs/objectives/reduce-churn-objective.md <<'MD'
# Reduce Preventable Churn

Customer success leaders want to reduce preventable churn by identifying
accounts with declining product usage, unresolved support friction, and renewal
risk before the next business review.
MD

cat > source_docs/use-cases/retention-risk-workflow.md <<'MD'
# Retention Risk Workflow

Customer success managers need a weekly workflow that ranks accounts by churn
risk, explains the main risk drivers, and suggests which accounts should be
contacted before renewal.
MD

cat > source_docs/signals/churn-risk-signal.txt <<'TXT'
Daily customer health note from April 18, 2026 at 09:30.

Product usage is down for several priority accounts, unresolved support tickets
are increasing, and renewal conversations have slowed. The signal should help
retention teams detect preventable churn earlier.
TXT

cat > source_docs/products/customer-health-product.md <<'MD'
# Customer Health Signals Product

The product combines customer profile, subscription status, product usage,
support ticket volume, renewal date, campaign engagement, and churn-risk
signals. It is used by customer success and lifecycle marketing teams for
retention planning.
MD

## Review The Portfolio Source Documents

Before building the portfolio, inspect the source material. The builder is not starting from an empty prompt; it is using these lane-specific business notes as evidence.

### Before You Run This

Each source lane contributes a different kind of portfolio input:

- `objectives/` explains the business goal.
- `use-cases/` explains the workflow or user need.
- `signals/` provides observed evidence or market/customer signals.
- `products/` describes the product idea or existing product material.

What the code does:

- Loops through the four source lanes.
- Prints the first file name in each lane.
- Shows a short preview so you know what the portfolio builder will read.

In [6]:
# Preview one source document from each portfolio lane.
import os
from pathlib import Path

base = Path("/content/odp-portfolio-workspace/source_docs")
for lane in ["objectives", "use-cases", "signals", "products"]:
    files = sorted((base / lane).glob("*"))
    if not files:
        print(f"\n## {lane}: no files found")
        continue
    path = files[0]
    print(f"\n## {lane}: {path.name}")
    print(path.read_text()[:700].strip())


## objectives: reduce-churn-objective.md
# Reduce Preventable Churn

Customer success leaders want to reduce preventable churn by identifying
accounts with declining product usage, unresolved support friction, and renewal
risk before the next business review.

## use-cases: retention-risk-workflow.md
# Retention Risk Workflow

Customer success managers need a weekly workflow that ranks accounts by churn
risk, explains the main risk drivers, and suggests which accounts should be
contacted before renewal.

## signals: churn-risk-signal.txt
Daily customer health note from April 18, 2026 at 09:30.

Product usage is down for several priority accounts, unresolved support tickets
are increasing, and renewal conversations have slowed. The signal should help
retention teams detect preventable churn earlier.

## products: customer-health-product.md
# Customer Health Signals Product

The product combines customer profile, subscription status, product usage,
support ticket volume, renewal date, c

## Check The Source Files

Use this quick check before building to see what the notebook created.

In [7]:
!find source_docs -maxdepth 3 -type f | sort

source_docs/objectives/reduce-churn-objective.md
source_docs/products/customer-health-product.md
source_docs/signals/churn-risk-signal.txt
source_docs/use-cases/retention-risk-workflow.md


## Run Portfolio Build

This creates a portfolio workspace with ODPC, ODPS, ODPG, HTML, and report artifacts.

### Before You Run This

This is the main orchestration command for the portfolio workflow. Instead of manually generating fragments, building a catalog, building a graph, and rendering HTML one step at a time, `portfolio build` combines those actions into one workflow.

The command uses the four source lanes you just created:

- `--objectives source_docs/objectives/` gives the builder business goals.
- `--use-cases source_docs/use-cases/` gives it workflows or user needs.
- `--signals source_docs/signals/` gives it observed evidence and prioritization signals.
- `--products source_docs/products/` gives it product notes or product descriptions.

The remaining parameters control the build:

- `--title "Customer Intelligence Portfolio"` names the browser review page.
- `--output portfolio/` tells the SDK where to write the generated workspace.
- `--provider claude` selects the configured LLM provider.
- `--model claude-sonnet-4-5` selects the model used for generation.

What the code does:

- Reads the four source lanes.
- Uses the LLM provider to turn source material into portfolio artifacts.
- Writes ODPC catalog files, ODPS product files, ODPG graph files, reports, and HTML output into `portfolio/`.
- Creates the first complete portfolio package for human and agent review.

In [8]:
import os

if os.environ.get("ANTHROPIC_API_KEY"):
    !open-data-products portfolio build \
      --objectives source_docs/objectives/ \
      --use-cases source_docs/use-cases/ \
      --signals source_docs/signals/ \
      --products source_docs/products/ \
      --title "Customer Intelligence Portfolio" \
      --output portfolio/ \
      --provider claude \
      --model claude-sonnet-4-5
else:
    print("Skipped: add ANTHROPIC_API_KEY to Colab secrets, rerun the key cell, then rerun portfolio build.")

Workspace: portfolio
HTML: portfolio/index.html
Validation mode: warn
Created: 10
Updated: 0


## Inspect Generated Folders

Look for `portfolio/index.html`, `portfolio/portfolio.yaml`, `portfolio/odpc/catalog.yaml`, `portfolio/odpc/fragments/`, `portfolio/odpg/graph.yaml`, and `portfolio/odps/products/`.

### Before You Run This

Do not worry if there are many files. The most important pattern is: HTML is for human review, YAML is for standards-based automation, and version/report files help explain what happened during the build.

What the code does:

- Lists generated files inside the `portfolio/` folder.
- Limits the depth so the output is useful without flooding the notebook.
- Sorts the list so the folder structure is easier to scan.


In [9]:
import os
from pathlib import Path

if Path("portfolio/index.html").exists():
    !find portfolio -maxdepth 3 -type f | sort
else:
    print("No portfolio output found yet. Run the portfolio build cell after adding ANTHROPIC_API_KEY.")

portfolio/index.html
portfolio/odpc/catalog.yaml
portfolio/odpc/fragments/business_objective_OBJ-REDUCE-PREVENTABLE-CHURN.yaml
portfolio/odpc/fragments/product_reference_PR-CUSTOMER-HEALTH-SIGNALS.yaml
portfolio/odpc/fragments/signal_SIG-CHURN-RISK-SIGNAL.yaml
portfolio/odpc/fragments/use_case_UC-RETENTION-RISK-WORKFLOW.yaml
portfolio/odpg/graph.yaml
portfolio/odps/products/customer-health-signals.yaml
portfolio/portfolio-state.yaml
portfolio/portfolio.yaml


## Review Catalog And Graph Files

These commands explain the workspace and validate the generated catalog and graph artifacts.

In [10]:
from pathlib import Path

if Path("portfolio/odpc/catalog.yaml").exists() and Path("portfolio/odpg/graph.yaml").exists():
    !open-data-products portfolio explain portfolio/
    !open-data-products validate portfolio/odpc/catalog.yaml
    !open-data-products validate portfolio/odpg/graph.yaml
else:
    print("Catalog and graph files are missing. Run the portfolio build cell first.")


Workspace: portfolio
HTML: portfolio/index.html
Validation mode: warn
Product references: 1
✓ Loaded ODPC document: portfolio/odpc/catalog.yaml
✓ Detected kind: Catalog
✓ Detected version: 1.0
✓ Schema validation passed
✓ ODPC validation passed

Validation successful!
✓ Loaded ODPG document: portfolio/odpg/graph.yaml
✓ Detected kind: Graph
✓ Detected version: 1.0
✓ Schema validation passed
✓ ODPG validation passed

Validation successful!


## Package And Review The HTML Output

The portfolio HTML is the main human review page. This cell packages the full `portfolio/` folder and downloads it from Colab so you can inspect the browser view locally.

After the download finishes:

1. Unzip `portfolio-review.zip` on your computer.
2. Open `portfolio/index.html` in a browser.
3. Keep the YAML files beside it; the HTML and YAML belong together as one review package.

In [11]:
import os
from pathlib import Path

if Path("portfolio/index.html").exists():
    !zip -r portfolio-review.zip portfolio
    from google.colab import files
    files.download("portfolio-review.zip")
else:
    print("No portfolio HTML found yet. Run the portfolio build cell first.")


  adding: portfolio/ (stored 0%)
  adding: portfolio/odps/ (stored 0%)
  adding: portfolio/odps/products/ (stored 0%)
  adding: portfolio/odps/products/customer-health-signals.yaml (deflated 73%)
  adding: portfolio/portfolio-state.yaml (deflated 51%)
  adding: portfolio/index.html (deflated 80%)
  adding: portfolio/odpg/ (stored 0%)
  adding: portfolio/odpg/graph.yaml (deflated 63%)
  adding: portfolio/portfolio.yaml (deflated 43%)
  adding: portfolio/odpc/ (stored 0%)
  adding: portfolio/odpc/catalog.yaml (deflated 56%)
  adding: portfolio/odpc/fragments/ (stored 0%)
  adding: portfolio/odpc/fragments/signal_SIG-CHURN-RISK-SIGNAL.yaml (deflated 33%)
  adding: portfolio/odpc/fragments/business_objective_OBJ-REDUCE-PREVENTABLE-CHURN.yaml (deflated 31%)
  adding: portfolio/odpc/fragments/use_case_UC-RETENTION-RISK-WORKFLOW.yaml (deflated 30%)
  adding: portfolio/odpc/fragments/product_reference_PR-CUSTOMER-HEALTH-SIGNALS.yaml (deflated 42%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## What You Learned

- Portfolio build combines source lanes into one workspace.
- The workspace includes ODPC, ODPS, ODPG, HTML, and report artifacts.
- The browser view is for review, while YAML files remain agent-ready.

## Next Lesson

Continue to Lecture 20: Update the portfolio and review version history.

# Lecture 20: Update the Portfolio and Review Version History

A portfolio is a living artifact. Source material changes, new signals appear, and product drafts improve over time. This lesson shows how to refresh a workspace and review previous versions.

Continue from the same workspace folder you created in Lecture 19:

```text
/content/odp-portfolio-workspace
```

All commands in this lesson assume the current folder contains both `source_docs/` and `portfolio/`.

## Check Runtime State

Colab runtimes can restart between lessons. Before continuing, confirm that the workspace, source lanes, and generated portfolio still exist. If this check reports missing files, rerun Lecture 19 from the workspace creation cell through portfolio build.

In [13]:
import os
from pathlib import Path

workspace = Path("/content/odp-portfolio-workspace")
checks = {
    "workspace folder": workspace,
    "source lanes": workspace / "source_docs",
    "portfolio output": workspace / "portfolio",
    "portfolio HTML": workspace / "portfolio" / "index.html",
}

missing = [name for name, path in checks.items() if not path.exists()]
if missing:
    print("Missing: " + ", ".join(missing))
    print("Rerun Lecture 19 setup and portfolio build before continuing.")
else:
    os.chdir(workspace)
    print("Workspace is ready. Continue with Lecture 20.")
    print("Current folder:", Path.cwd())

Workspace is ready. Continue with Lecture 20.
Current folder: /content/odp-portfolio-workspace


## Add Or Change Source Material

A portfolio should not be treated as a one-time export. Business priorities change, new signals appear, and reviewers need to compare current and previous outputs. Version history makes those changes easier to review and govern.

Next, we add a new signal, refresh the portfolio, and inspect versioned outputs.

### Before You Add This Source

This new signal simulates a realistic portfolio update: new evidence arrives after the first build. Instead of rebuilding the course from scratch conceptually, we add one source document and let the refresh workflow update the portfolio.

What the code does:

- Writes a new text file into `source_docs/signals/`.
- Leaves the original source files in place.
- Gives the next refresh command a concrete change to process.

In [14]:
%%bash
cat > source_docs/signals/support-pressure-signal.txt <<'TXT'
Support operations note from April 19, 2026 at 10:15.

Priority accounts with unresolved tickets are waiting longer for first response.
Customer success teams want this signal linked to retention review workflows.
TXT

## Refresh Changed Sources - soft approach

**Soft refresh scans saved source lanes and sends changed or new source files to the LLM. Existing unchanged artifacts are preserved where possible.**

### Before You Run This

Refresh is for normal portfolio updates. It looks at the saved source lanes and processes changed or new source files while preserving unchanged artifacts where possible.

What the code does:

- Runs a normal portfolio refresh against the existing `portfolio/` workspace.
- Uses the configured provider and model to process changed or new source material.


In [15]:
import os
from pathlib import Path

if not Path("portfolio/index.html").exists():
    print("No portfolio output found. Rerun Lecture 19 before refreshing.")
elif not os.environ.get("ANTHROPIC_API_KEY"):
    print("Skipped: add ANTHROPIC_API_KEY to Colab secrets, rerun the key cell, then rerun refresh.")
else:
    !open-data-products portfolio refresh portfolio/ \
      --provider claude \
      --model claude-sonnet-4-5

Workspace: portfolio
HTML: portfolio/index.html
Validation mode: warn
Created: 1
Updated: 5


## Force Full Reprocessing

Use `--all-sources` when the full evidence set should be reprocessed.

### Before You Run This

A full reprocess is heavier than a normal refresh. Use it when you want the complete source set reconsidered, such as after changing model choice, prompts, or the overall interpretation of the portfolio.

What the code does:

- Runs refresh with `--all-sources`.
- Reprocesses the full source set instead of only changed or new source files.
- Uses the same provider and model as the first build.


In [ ]:
import os
from pathlib import Path

if not Path("portfolio/index.html").exists():
    print("No portfolio output found. Rerun Lecture 19 before full reprocessing.")
elif not os.environ.get("ANTHROPIC_API_KEY"):
    print("Skipped: add ANTHROPIC_API_KEY to Colab secrets, rerun the key cell, then rerun full reprocessing.")
else:
    !open-data-products portfolio refresh portfolio/ \
      --all-sources \
      --provider claude \
      --model claude-sonnet-4-5

## Sync Edited YAML Without An LLM

Use sync after directly editing ODPC fragments, ODPS products, or graph YAML.

### Before You Run This

Sync is the non-LLM path. Use it when you manually edit YAML and only need the browser view or derived files rebuilt from those existing artifacts.

What the code does:

- Rebuilds derived portfolio outputs from existing YAML.
- Does not call an LLM.
- Is useful after manual YAML edits.


In [16]:
from pathlib import Path

if Path("portfolio").exists():
    !open-data-products portfolio sync portfolio/
else:
    print("No portfolio folder found. Rerun Lecture 19 before syncing.")


Workspace: portfolio
HTML: portfolio/index.html
Validation mode: warn
Created: 0
Updated: 3


## Review Version History

Successful builds and refreshes snapshot previous portfolio outputs. The latest `index.html` includes links to available versions so reviewers can compare current and previous portfolio pages.

### Before You Review Version History

Version history helps reviewers compare the current portfolio with earlier outputs. This is useful when source material changes, when reviewers ask what changed, or when governance needs an audit trail.

What the code does:

- Lists files under `portfolio/versions/`.
- Shows the saved snapshot structure created by build and refresh operations.
- Helps you confirm that previous portfolio outputs were preserved.

In [17]:
from pathlib import Path

if Path("portfolio/versions").exists():
    !find portfolio/versions -maxdepth 2 -type f | sort
else:
    print("No version snapshots found yet. Run portfolio build and at least one refresh first.")

portfolio/versions/2026-06-25T13-22-13Z/index.html
portfolio/versions/2026-06-25T13-22-13Z/portfolio.yaml
portfolio/versions/2026-06-25T13-22-13Z/report.json
portfolio/versions/2026-06-25T13-24-38Z/index.html
portfolio/versions/2026-06-25T13-24-38Z/portfolio.yaml
portfolio/versions/2026-06-25T13-24-38Z/report.json


### Lets review that the update happened

Run the below code to download full portfolio package again.

In [18]:
import os
from pathlib import Path

if Path("portfolio/index.html").exists():
    !zip -r portfolio-update-review.zip portfolio
    from google.colab import files
    files.download("portfolio-update-review.zip")
else:
    print("No portfolio HTML found yet. Run the portfolio build cell first.")

  adding: portfolio/ (stored 0%)
  adding: portfolio/odps/ (stored 0%)
  adding: portfolio/odps/products/ (stored 0%)
  adding: portfolio/odps/products/customer-health-signals.yaml (deflated 73%)
  adding: portfolio/portfolio-state.yaml (deflated 52%)
  adding: portfolio/index.html (deflated 80%)
  adding: portfolio/odpg/ (stored 0%)
  adding: portfolio/odpg/graph.yaml (deflated 65%)
  adding: portfolio/portfolio.yaml (deflated 76%)
  adding: portfolio/versions/ (stored 0%)
  adding: portfolio/versions/2026-06-25T13-22-13Z/ (stored 0%)
  adding: portfolio/versions/2026-06-25T13-22-13Z/index.html (deflated 80%)
  adding: portfolio/versions/2026-06-25T13-22-13Z/portfolio.yaml (deflated 43%)
  adding: portfolio/versions/2026-06-25T13-22-13Z/report.json (deflated 69%)
  adding: portfolio/versions/2026-06-25T13-24-38Z/ (stored 0%)
  adding: portfolio/versions/2026-06-25T13-24-38Z/index.html (deflated 80%)
  adding: portfolio/versions/2026-06-25T13-24-38Z/portfolio.yaml (deflated 70%)
  addi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## What You Learned

- Refresh processes changed and new source documents by default.
- `--all-sources` forces full reprocessing.
- `portfolio sync` rebuilds browser output from edited YAML without an LLM.
- Version snapshots support review and governance over time.

## Next Lesson

Continue to Lecture 21: How to localize a portfolio.

# Lecture 21: How to Localize a Portfolio

Portfolio localization creates translated static HTML review pages for regional stakeholders. It does not change the canonical ODPC, ODPS, or ODPG YAML files.

Use this lesson after you have a working portfolio workspace from the previous lessons.

Continue from the same workspace folder you created in Lecture 19:

```text
/content/odp-portfolio-workspace
```

Localization writes files into `portfolio/` inside that workspace. Do not download individual HTML files before running localization; keep `index.html`, localized pages, and `portfolio-i18n.yaml` together.

### Beginning of Localization

Localization changes the human-facing review experience while keeping the canonical YAML artifacts as the source of truth. It applies to the active portfolio page and does not retroactively translate older `portfolio/versions/` snapshots. If a translated record of a specific version is needed, localize while that version is current and preserve the zipped portfolio package.

Next, we localize the active portfolio page and package the localized review output.

## Start From A Rendered Portfolio

You need a portfolio workspace with a current `index.html`. If the page does not exist yet, render it from the current YAML artifacts.

In [19]:
from pathlib import Path

if Path("portfolio").exists():
    if Path("portfolio/index.html").exists():
        print("Found portfolio/index.html")
    else:
        print("No current HTML page found. Rendering from existing YAML artifacts.")
    !open-data-products portfolio render portfolio/
else:
    print("No portfolio folder found. Rerun Lecture 19 before localization.")


Found portfolio/index.html
Workspace: portfolio
HTML: portfolio/index.html
Validation mode: warn
Created: 0
Updated: 0


## Choose Target Languages

`portfolio localize` accepts BCP 47 language tags. Start with one or two languages while learning the workflow:

```text
fi   Finnish
sv   Swedish
ar   Arabic
vi   Vietnamese
```

Arabic is useful for checking right-to-left page rendering. Finnish and Swedish are useful for Nordic stakeholder review examples.

## Run Localization

Use a configured LLM provider to translate the visible HTML strings. The command reads the existing portfolio HTML, translates human-facing text, and writes localized pages beside the main `index.html`.

### Before You Run This

Localization translates the human-facing review page. It should preserve the structure, identifiers, and relationships that make the portfolio reliable for automation and governance.

What the code does:

- Runs localization for Finnish and Swedish.
- Uses the provider and model to translate visible portfolio page text.
- Writes localized HTML pages and translation data beside the main `index.html`.


In [20]:
import os
from pathlib import Path

if not Path("portfolio/index.html").exists():
    print("No rendered portfolio page found. Run the render check above or rerun Lecture 18.")
elif not os.environ.get("ANTHROPIC_API_KEY"):
    print("Skipped: add ANTHROPIC_API_KEY to Colab secrets, rerun the key cell, then rerun localization.")
else:
    !open-data-products portfolio localize portfolio/ \
      --languages "vi,ar" \
      --provider claude \
      --model claude-sonnet-4-5

Workspace: portfolio
HTML: {'en': 'portfolio/index.html', 'vi': 'portfolio/index.vi.html', 'ar': 'portfolio/index.ar.html'}
Validation mode: warn
Created: 3
Updated: 1


## Review The Outputs

Check that the localized pages and translation file exist. `portfolio-i18n.yaml` stores the translated strings used to render localized HTML pages. It is generated from the portfolio view, not from editing the canonical YAML artifacts.

### Before You Review These Outputs

Localization creates extra human-facing files beside the main portfolio page. The important file to understand is `portfolio-i18n.yaml`: it stores translated strings used to render localized HTML pages, but it is not the canonical business artifact.

What the code does:

- Checks that the Vietname and Arabic HTML pages exist.
- Checks that `portfolio-i18n.yaml` exists.
- Confirms that localization produced a package of related files, not just a single translated page.

In [21]:
from pathlib import Path

expected = [
    Path("portfolio/index.vi.html"),
    Path("portfolio/index.ar.html"),
    Path("portfolio/portfolio-i18n.yaml"),
]
missing = [str(path) for path in expected if not path.exists()]
if missing:
    print("Missing localized output: " + ", ".join(missing))
    print("Run localization after adding ANTHROPIC_API_KEY.")
else:
    !ls portfolio/index.vi.html
    !ls portfolio/index.ar.html
    !ls portfolio/portfolio-i18n.yaml

portfolio/index.vi.html
portfolio/index.ar.html
portfolio/portfolio-i18n.yaml


## Package And Review The Localized Pages

The localized pages should be reviewed in a browser, not only listed in Colab. This cell packages the full `portfolio/` folder again because localized HTML pages depend on files beside them.

After the download finishes:

1. Unzip `portfolio-localized-review.zip` on your computer.
2. Open `portfolio/index.vi.html` and `portfolio/index.ar.html` in a browser.
3. Compare them with `portfolio/index.html`.
4. Keep `portfolio/portfolio-i18n.yaml` with the package so translated strings remain traceable.

In [22]:
from pathlib import Path

if Path("portfolio/index.vi.html").exists() and Path("portfolio/index.ar.html").exists():
    !zip -r portfolio-localized-review.zip portfolio
    from google.colab import files
    files.download("portfolio-localized-review.zip")
else:
    print("Localized pages are missing. Run localization before downloading the localized review package.")


  adding: portfolio/ (stored 0%)
  adding: portfolio/index.ar.html (deflated 80%)
  adding: portfolio/odps/ (stored 0%)
  adding: portfolio/odps/products/ (stored 0%)
  adding: portfolio/odps/products/customer-health-signals.yaml (deflated 73%)
  adding: portfolio/portfolio-state.yaml (deflated 52%)
  adding: portfolio/index.html (deflated 80%)
  adding: portfolio/portfolio-i18n.yaml (deflated 73%)
  adding: portfolio/odpg/ (stored 0%)
  adding: portfolio/odpg/graph.yaml (deflated 65%)
  adding: portfolio/portfolio.yaml (deflated 76%)
  adding: portfolio/index.vi.html (deflated 80%)
  adding: portfolio/versions/ (stored 0%)
  adding: portfolio/versions/2026-06-25T13-22-13Z/ (stored 0%)
  adding: portfolio/versions/2026-06-25T13-22-13Z/index.html (deflated 80%)
  adding: portfolio/versions/2026-06-25T13-22-13Z/portfolio.yaml (deflated 43%)
  adding: portfolio/versions/2026-06-25T13-22-13Z/report.json (deflated 69%)
  adding: portfolio/versions/2026-06-25T13-24-38Z/ (stored 0%)
  adding:

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Keep YAML As The Source Of Truth

Localization is for human-facing review pages. The canonical files remain:

```text
portfolio/portfolio.yaml
portfolio/odpc/catalog.yaml
portfolio/odpc/fragments/*.yaml
portfolio/odpg/graph.yaml
portfolio/odps/products/*.yaml
```

Agents, scripts, validation commands, and version control should keep using those YAML files as the source of truth.

### Before You Run This

Localized HTML helps people review the portfolio in another language, but the canonical YAML files remain the authoritative artifacts for agents, scripts, validation, and version control.

What the code does:

- Finds YAML files in the active portfolio package.
- Skips old version snapshot folders so the list focuses on the current canonical artifacts.
- Sorts the result for easier review.


In [ ]:
from pathlib import Path

if Path("portfolio").exists():
    !find portfolio -path "*/versions/*" -prune -o -name "*.yaml" -print | sort
else:
    print("No portfolio folder found. Rerun Lecture 18 before reviewing YAML artifacts.")

## Troubleshooting

If localization takes too long with a local model, try fewer languages first. If a command pasted into `zsh` behaves strangely, check that every line continuation backslash is the final character on its line. A space after `\` breaks the command.

In [ ]:
# Optional local-model localization example.
# !open-data-products portfolio localize portfolio/ \
#   --languages "fi" \
#   --provider ollama \
#   --model qwen2.5

## Strict Validation Option

If automation should fail on validation issues, add `--strict-validation`.

In [ ]:
# Optional strict validation example.
# !open-data-products portfolio localize portfolio/ \
#   --languages "fi,sv" \
#   --provider claude \
#   --model claude-sonnet-4-5 \
#   --strict-validation

## What You Learned

- `portfolio localize` creates translated static HTML review pages.
- Localization leaves ODPC, ODPS, and ODPG YAML artifacts unchanged.
- `portfolio-i18n.yaml` stores translated page strings.
- BCP 47 tags such as `fi`, `sv`, `ar`, and `vi` select target languages.
- Local models can work, but hosted providers are often better for longer multilingual review pages.

## Next Lesson

Continue to Lecture 22: Final review: human view and agent-ready YAML.

# Lecture 22: Final Review: Human View and Agent-Ready YAML

The portfolio workflow serves two audiences. The HTML view is for humans. The YAML files are for AI agents, automation, validation, and long-term version control.

Continue from the same workspace folder you created in Lecture 19:

```text
/content/odp-portfolio-workspace
```

### What We Just Covered

The final portfolio package serves two audiences. Humans review the HTML pages, including localized pages when available. Agents, scripts, validators, and version control use the YAML artifacts.

Next, we inspect both sides of the output and run the final validation commands.

## Human Review

Open:

```text
/content/odp-portfolio-workspace/portfolio/index.html
```

Review the portfolio as a human reviewer would:

- Check the overview: does the portfolio story make sense?
- Open product cards: are the product references understandable?
- Review artifact sections: do objectives, use cases, signals, and products connect logically?
- Review the graph section: do relationships match the source material?
- Check raw artifact links or YAML references when available.

If you localized the portfolio in Lecture 20, review those pages from the same workspace too:

```text
/content/odp-portfolio-workspace/portfolio/index.fi.html
/content/odp-portfolio-workspace/portfolio/index.sv.html
```

The goal is not only to see that files exist. The goal is to decide whether the generated portfolio is clear enough for stakeholders and structured enough for follow-up work.

## Agent-Ready YAML

Review the generated YAML files. These are the key files for agents, automation, validation, and version control.

### Before You Run This

This final review looks beyond the browser page. The YAML files are what another tool, workflow, or AI agent can consume later, so they need to be findable, valid, and consistent.

What the code does:

- Lists the current YAML artifacts again during final review.
- Excludes version snapshots so learners focus on the active catalog, graph, product, and portfolio files.


In [24]:
from pathlib import Path

if Path("portfolio").exists():
    !find portfolio -path "*/versions/*" -prune -o -name "*.yaml" -print | sort
else:
    print("No portfolio folder found. Rerun Lecture 18 before final YAML review.")

portfolio/odpc/catalog.yaml
portfolio/odpc/fragments/business_objective_OBJ-REDUCE-PREVENTABLE-CHURN.yaml
portfolio/odpc/fragments/product_reference_PR-CUSTOMER-HEALTH-SIGNALS.yaml
portfolio/odpc/fragments/signal_SIG-CHURN-RISK-SIGNAL.yaml
portfolio/odpc/fragments/signal_SIG-SUPPORT-PRESSURE.yaml
portfolio/odpc/fragments/use_case_UC-RETENTION-RISK-WORKFLOW.yaml
portfolio/odpg/graph.yaml
portfolio/odps/products/customer-health-signals.yaml
portfolio/portfolio-i18n.yaml
portfolio/portfolio-state.yaml
portfolio/portfolio.yaml


The important files are:

- `portfolio/portfolio.yaml`
- `portfolio/odpc/catalog.yaml`
- `portfolio/odpc/fragments/*.yaml`
- `portfolio/odpg/graph.yaml`
- `portfolio/odps/products/*.yaml`

## Validate The Artifacts

Portfolio commands default to warning mode for schema-invalid generated ODPS drafts so users can still review the browser output. Use `--strict-validation` when automation should fail on schema errors.

### Before You Run This

Validation is the final governance check in this workflow. It does not replace human judgment, but it catches structural problems before generated artifacts are reused or shared.

What the code does:

- Validates the ODPC catalog YAML.
- Validates the ODPG graph YAML.
- Runs `portfolio explain` to summarize the complete workspace for review.


In [23]:
from pathlib import Path

if Path("portfolio/odpc/catalog.yaml").exists() and Path("portfolio/odpg/graph.yaml").exists():
    !open-data-products validate portfolio/odpc/catalog.yaml
    !open-data-products validate portfolio/odpg/graph.yaml
    !open-data-products portfolio explain portfolio/
else:
    print("Catalog and graph files are missing. Rerun Lecture 18 before final validation.")


✓ Loaded ODPC document: portfolio/odpc/catalog.yaml
✓ Detected kind: Catalog
✓ Detected version: 1.0
✓ Schema validation passed
✓ ODPC validation passed

Validation successful!
✓ Loaded ODPG document: portfolio/odpg/graph.yaml
✓ Detected kind: Graph
✓ Detected version: 1.0
✓ Schema validation passed
✓ ODPG validation passed

Validation successful!
Workspace: portfolio
HTML: portfolio/index.html
Validation mode: warn
Product references: 1


## What You Learned

- HTML supports human portfolio review.
- YAML supports AI agents and automation.
- Final review checks both the browser experience and the generated YAML artifacts.
- Catalogs provide structure, graphs provide relationships, and validation supports governance.

## Next Lesson

Continue to Lecture 23: Wrap-up and next steps.

# Lecture 23: Wrap-up and Next Steps

You have completed the course path from SDK basics to a connected, reviewable, agent-ready data product portfolio workflow.

### What We Just Covered

The course path moved from SDK basics to a connected portfolio workflow: setup, validation, vocabulary, LLM configuration, generation, fragments, graphs, catalogs, portfolio build, version history, localization, and final review.

Next, we close by connecting those skills to real portfolio work after the course.

## What You Can Do Now

You can now use the Open Data Products SDK to:

- install and run the CLI
- validate and explain standards files
- use ODPV vocabulary helpers
- configure local and online LLM providers
- generate full ODPS product drafts
- generate ODPC fragments
- build and inspect ODPG graph relationships
- build ODPC catalogs
- create, refresh, sync, localize, render, and review a portfolio workspace
- inspect both browser output and machine-readable YAML artifacts

## How The Pieces Fit

The individual SDK commands give you precise control over one artifact or one step. The portfolio builder combines those capabilities into a repeatable workflow for real portfolio work.

ODPS describes data products. ODPC organizes portfolio catalog objects. ODPG describes relationships. ODPV keeps language consistent.

Together they create a practical pattern: business intent and source material can become human-reviewable HTML and agent-ready YAML.

## Portfolio As A Workflow

You can now treat the portfolio as a workflow, not only as a set of files. That is important because real portfolio work is repeated: source material changes, reviewers ask for updates, regional stakeholders need localized pages, and agents need clean YAML artifacts.

The portfolio command group gives you a repeatable sequence:

```bash
open-data-products portfolio build \
  --objectives source_docs/objectives/ \
  --use-cases source_docs/use-cases/ \
  --signals source_docs/signals/ \
  --products source_docs/products/ \
  --output portfolio/

open-data-products portfolio refresh portfolio/
open-data-products portfolio sync portfolio/
open-data-products portfolio localize portfolio/ \
  --languages "fi,sv" \
  --provider claude \
  --model claude-sonnet-4-5
open-data-products portfolio render portfolio/
open-data-products portfolio explain portfolio/
```

This matters because the SDK keeps the workflow grounded in artifacts:

- source lanes keep business objectives, use cases, signals, and product briefs organized;
- ODPC catalogs describe the portfolio structure;
- ODPS product YAML keeps data product details machine-readable;
- ODPG graphs describe relationships between portfolio objects;
- HTML gives people a reviewable browser view;
- localization creates regional review pages without changing canonical YAML;
- version snapshots support governance and change review over time.

In other words, the portfolio workflow turns scattered source material into a repeatable operating model: build, review, update, localize, validate, explain, and keep improving.

### What We Just Covered

The portfolio workflow is more than a command sequence. It is an operating model: build, review, update, localize, validate, explain, and keep improving as source material and stakeholder needs change.

Next, we summarize how the portfolio command group keeps that workflow grounded in source lanes, catalogs, product YAML, graphs, HTML, localization, and version snapshots.

## Coming Next: ODPR Workflow Recipes

ODPR, the Open Data Product Recipe Specification, is in development now and is published as a draft specification. Because of that, ODPR is not applied as a hands-on standard in this Masterclass. The portfolio workflow you ran in this notebook uses the current SDK portfolio commands directly.

ODPR is the next layer to watch: workflow recipes for repeatable SDK runs. Where the portfolio command group gives you a built-in workflow, ODPR is intended to let teams define their own workflows on top of the SDK.

An ODPR-style recipe could define:

- the ordered SDK steps to run;
- which provider or model to use for each step;
- input and output folders;
- validation gates;
- context formats such as YAML, TOON, or GCF;
- review and localization policy.

For example, a future recipe could capture a release review workflow:

```yaml
recipes:
  release-portfolio-review:
    description: Refresh, localize, render, and explain the release portfolio.
    provider: claude
    steps:
      - command: portfolio.refresh
        workspace: portfolio/
      - command: portfolio.localize
        workspace: portfolio/
        languages: fi,sv
      - command: portfolio.render
        workspace: portfolio/
      - command: portfolio.explain
        workspace: portfolio/
```

The value is simple: instead of copying command sequences between terminals, notebooks, CI jobs, and team documents, a project can name the workflow and run it consistently. ODPR is the direction for making those repeatable workflows explicit, portable, and easier for both humans and AI agents to follow.

Read more from the ODPR draft specification: [Open Data Product Recipe Specification v1.0](https://opendataproducts.org/odpr-v1.0/).

### What We Just Covered

ODPR is the next workflow layer to watch. It is still in draft and is not applied hands-on in this Masterclass, but the idea is important: repeatable SDK workflows can be expressed as named recipes instead of copied manually between notebooks, terminals, CI jobs, and team documents.

Next, we look at a small example of what an ODPR-style recipe could represent.

## Apply This With Your Own Material

Start with source material from a real data product or portfolio idea:

- business objectives
- use cases
- market, operational, customer, quality, or usage signals
- product briefs, emails, transcripts, or governance notes

Put those files into the four source lanes:

```text
odp-portfolio-workspace/source_docs/objectives/
odp-portfolio-workspace/source_docs/use-cases/
odp-portfolio-workspace/source_docs/signals/
odp-portfolio-workspace/source_docs/products/
```

Run `portfolio build` from inside `odp-portfolio-workspace` to create the `portfolio/` workspace output. Review `portfolio/index.html` with people, then inspect the YAML files with agents, scripts, validation commands, or version control.

Then keep the portfolio alive by refreshing sources, syncing edited YAML, and using version snapshots during review. When the audience changes, localize the HTML pages without changing the canonical YAML artifacts.

## Reference Material

Use these references when you want to go deeper:

- [SDK README](https://github.com/Open-Data-Product-Initiative/odps-python/blob/main/README.md)
- [SDK API reference](https://github.com/Open-Data-Product-Initiative/odps-python/blob/main/docs/user/API.md)
- [SDK command guide](https://github.com/Open-Data-Product-Initiative/odps-python/blob/main/docs/user/commands.md)
- [Generation guide](https://github.com/Open-Data-Product-Initiative/odps-python/blob/main/docs/user/generation.md)
- [Portfolio development notes](https://github.com/Open-Data-Product-Initiative/odps-python/blob/main/docs/development/portfolio.md)
- [ODPS product specification](https://opendataproducts.org/v4.1/)
- [ODPC catalog specification](https://opendataproducts.org/odpc-v1.0/)
- [ODPG graph specification](https://opendataproducts.org/odpg-v1.0/)
- [ODPV vocabulary specification](https://opendataproducts.org/odpv-v1.0/)

## Thank You

Thank you for taking the course. If you apply these ideas in your own work, consider sharing your experience, lessons learned, and examples in a blog post or on LinkedIn. Your notes can help other data, analytics, and AI practitioners understand how open data product standards can work in real projects.

You can also connect with me on [LinkedIn](https://ae.linkedin.com/in/jarkkomoilanen) for further discussion.